## Collecte de données

In [2]:
pip install wikipedia-api

Note: you may need to restart the kernel to use updated packages.


In [8]:
import wikipediaapi
import csv
import time
import pandas as pd
from datetime import datetime

In [11]:
# Configuration de l'API
wiki = wikipediaapi.Wikipedia(
    user_agent="EnglishAthleteScraper/1.0 (contact: your@email.com)",
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

def scrapper_avec_limite(categories_dict, max_per_cat=1000):
    dataset = []
    start_time = time.time()
    
    print(f"Démarrage du scraping (Limite: {max_per_cat} par catégorie)")
    print(f"Heure de début : {datetime.now().strftime('%H:%M:%S')}")
    print("-" * 60)

    for label, cat_name in categories_dict.items():
        print(f"\n[Secteur] : {label}")
        cat_page = wiki.page(f"Category:{cat_name}")
        
        if not cat_page.exists():
            print(f"  [!] Catégorie '{cat_name}' introuvable sur Wikipedia EN.")
            continue

        current_cat_count = 0
        members = cat_page.categorymembers.values()
        
        for p in members:
            # On s'arrête si on atteint la limite pour cette catégorie
            if current_cat_count >= max_per_cat:
                print(f"  [!] Limite de {max_per_cat} atteinte pour {label}.")
                break
                
            # On ne prend que les articles (Espace de nom 0)
            if p.ns == wikipediaapi.Namespace.MAIN:
                dataset.append({
                    "Name": p.title,
                    "Biography": p.summary,
                    "Sport_Gender": label,
                    "Nationality": "English"
                })
                current_cat_count += 1
                
                # Print de suivi tous les 50 articles
                if current_cat_count % 50 == 0:
                    elapsed = time.time() - start_time
                    print(f"    > {label} : {current_cat_count}/{max_per_cat} récupérés | Temps : {int(elapsed)}s")

        print(f"  [✓] Terminé pour {label}. total récupéré : {current_cat_count}")

    print("-" * 60)
    print(f"Extraction terminée en {round((time.time() - start_time)/60, 2)} minutes.")
    return pd.DataFrame(dataset)

# --- Catégories Anglaises (Ciblées par Genre) ---
target_categories = {
    "Football_Men": "English_men's_footballers",
    "Football_Women": "English_women's_footballers",
    "Rugby_Men": "English_rugby_union_players",
    "Rugby_Women": "English_women's_rugby_union_players",
    "Gymnastics_Men": "English_male_artistic_gymnasts",
    "Gymnastics_Women": "English_female_artistic_gymnasts",
    "Figure_Skating_Men": "English_male_figure_skaters",
    "Figure_Skating_Women": "English_female_figure_skaters"
}

# Lancement avec la limite de 1000
df_final = scrapper_avec_limite(target_categories, max_per_cat=1000)

# Sauvegarde finale
if not df_final.empty:
    df_final.drop_duplicates(subset=['Name'], inplace=True)
    df_final.to_csv("english_athletes_1000_limit.csv", index=False, encoding='utf-8')
    print(f"\nFichier généré : english_athletes_1000_limit.csv")
    print(f"Nombre total de lignes : {len(df_final)}")

Démarrage du scraping (Limite: 1000 par catégorie)
Heure de début : 16:30:26
------------------------------------------------------------

[Secteur] : Football_Men
    > Football_Men : 50/1000 récupérés | Temps : 25s
    > Football_Men : 100/1000 récupérés | Temps : 36s
    > Football_Men : 150/1000 récupérés | Temps : 47s
    > Football_Men : 200/1000 récupérés | Temps : 58s
    > Football_Men : 250/1000 récupérés | Temps : 75s
    > Football_Men : 300/1000 récupérés | Temps : 92s
    > Football_Men : 350/1000 récupérés | Temps : 108s
    > Football_Men : 400/1000 récupérés | Temps : 122s
    > Football_Men : 450/1000 récupérés | Temps : 137s
    > Football_Men : 500/1000 récupérés | Temps : 149s
    > Football_Men : 550/1000 récupérés | Temps : 161s
    > Football_Men : 600/1000 récupérés | Temps : 176s
    > Football_Men : 650/1000 récupérés | Temps : 195s
    > Football_Men : 700/1000 récupérés | Temps : 212s
    > Football_Men : 750/1000 récupérés | Temps : 229s
    > Football_Men

In [22]:
import os

wiki = wikipediaapi.Wikipedia(
    user_agent="EnglishAthleteScraper/1.3 (contact: your@email.com)",
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

def completer_dataset(categories_dict, filename="english_athletes_1000_limit.csv", max_per_cat=1000):
    # 1. Chargement
    if os.path.exists(filename):
        df_existant = pd.read_csv(filename)
        df_existant.columns = [c.strip() for c in df_existant.columns]
        df_existant['Sport_Gender'] = df_existant['Sport_Gender'].astype(str).str.strip()
        df_existant['Name'] = df_existant['Name'].astype(str).str.strip()
        print(f"Chargement réussi : {len(df_existant)} lignes.")
    else:
        df_existant = pd.DataFrame(columns=["Name", "Biography", "Sport_Gender", "Nationality"])
        print("Nouveau fichier créé.")

    nouveaux_records = []
    noms_deja_presents = set(df_existant["Name"].tolist())

    # 2. Une seule boucle pour tout traiter
    for label, cat_name in categories_dict.items():
        label_propre = label.strip()
        
        # Comptage actuel
        deja_la = len(df_existant[df_existant["Sport_Gender"] == label_propre])
        a_recuperer = max_per_cat - deja_la
        
        if a_recuperer <= 0:
            print(f"[OK] {label_propre} est complet ({deja_la}/{max_per_cat}).")
            continue
            
        print(f"\n[Secteur] : {label_propre} ({deja_la} en stock, cherche {a_recuperer} de plus)")
        
        # On essaie de récupérer la page de la catégorie
        cat_page = wiki.page(f"Category:{cat_name}")
        
        if not cat_page.exists():
            print(f"  [!] ERREUR : La catégorie '{cat_name}' est toujours introuvable.")
            continue

        current_added = 0
        members = cat_page.categorymembers.values()
        
        for p in members:
            if current_added >= a_recuperer: break
            
            # CAS 1 : C'est un article direct
            if p.ns == wikipediaapi.Namespace.MAIN:
                nom = p.title.strip()
                if nom not in noms_deja_presents:
                    nouveaux_records.append({
                        "Name": nom,
                        "Biography": p.summary[:3000], # On limite pour éviter les CSV trop lourds
                        "Sport_Gender": label_propre,
                        "Nationality": "English"
                    })
                    noms_deja_presents.add(nom)
                    current_added += 1

            # CAS 2 : C'est une sous-catégorie (ex: Joueuses internationales)
            elif p.ns == wikipediaapi.Namespace.CATEGORY:
                for sub_p in p.categorymembers.values():
                    if current_added >= a_recuperer: break
                    if sub_p.ns == wikipediaapi.Namespace.MAIN:
                        nom_sub = sub_p.title.strip()
                        if nom_sub not in noms_deja_presents:
                            nouveaux_records.append({
                                "Name": nom_sub, 
                                "Biography": sub_p.summary[:3000], 
                                "Sport_Gender": label_propre, 
                                "Nationality": "English"
                            })
                            noms_deja_presents.add(nom_sub)
                            current_added += 1
        
        print(f"  [+] Ajoutés pour {label_propre} : {current_added}")

    # 3. Fusion et Sauvegarde
    if nouveaux_records:
        df_nouveau = pd.DataFrame(nouveaux_records)
        df_final = pd.concat([df_existant, df_nouveau], ignore_index=True)
        df_final.to_csv(filename, index=False, encoding='utf-8')
        print(f"\n[TERMINE] Dataset mis à jour. Total final : {len(df_final)} lignes.")
    else:
        print("\n[INFO] Rien à ajouter. Toutes les catégories disponibles ont été explorées.")

# --- NOMS DE CATÉGORIES TESTÉS ---
target_categories = {
    "Football_Men": "English men's footballers",
    "Football_Women": "English women's footballers",
    "Rugby_Men": "English rugby union players",
    # Variante la plus robuste pour le rugby féminin anglais :
    "Rugby_Women": "England women's international rugby union players",
    "Gymnastics_Men": "English male artistic gymnasts",
    "Gymnastics_Women": "English female artistic gymnasts",
    "Figure_Skating_Men": "English male figure skaters",
    "Figure_Skating_Women": "English female figure skaters"
}

completer_dataset(target_categories)

Chargement réussi : 2963 lignes.
[OK] Football_Men est complet (1000/1000).

[Secteur] : Football_Women (660 en stock, cherche 340 de plus)
  [+] Ajoutés pour Football_Women : 0
[OK] Rugby_Men est complet (1000/1000).

[Secteur] : Rugby_Women (0 en stock, cherche 1000 de plus)
  [+] Ajoutés pour Rugby_Women : 125

[Secteur] : Gymnastics_Men (107 en stock, cherche 893 de plus)
  [+] Ajoutés pour Gymnastics_Men : 0

[Secteur] : Gymnastics_Women (118 en stock, cherche 882 de plus)
  [+] Ajoutés pour Gymnastics_Women : 0

[Secteur] : Figure_Skating_Men (38 en stock, cherche 962 de plus)
  [+] Ajoutés pour Figure_Skating_Men : 0

[Secteur] : Figure_Skating_Women (40 en stock, cherche 960 de plus)
  [+] Ajoutés pour Figure_Skating_Women : 0

[TERMINE] Dataset mis à jour. Total final : 3088 lignes.


In [23]:
df = pd.read_csv('english_athletes_1000_limit.csv')
df

,Name,Biography,Sport_Gender,Nationality
0,Arthur Aaron (footballer),Arthur Frederick Aaron (21 September 1885 – 10...,Football_Men,English
1,Max Aarons,Maximillian James Aarons (born 4 January 2000)...,Football_Men,English
2,Rolando Aarons,Rolando Aarons (born 16 November 1995) is a pr...,Football_Men,English
3,Thelo Aasgaard,"Thelonious ""Thelo"" Gerard Aasgaard (born 2 May...",Football_Men,English
4,Godwin Abadaki,Godwin Olorunfemi Ebenmosi Abadaki (born 12 Oc...,Football_Men,English
...,...,...,...,...
3083,Danielle Waterman,"Danielle Sian ""Nolli"" Waterman (born 20 Januar...",Rugby_Women,English
3084,Bo Westcombe-Evans,Boudica Willow Westcombe-Evans (born 18 August...,Rugby_Women,English
3085,Kay Wilson,Kay Wilson (born 19 September 1991) is an Engl...,Rugby_Women,English
3086,Ella Wyrwas,Ella Maria Wyrwas (born 7 March 1999) is an En...,Rugby_Women,English
